In [8]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [9]:
from mrc.io import get_tomo, save_tomo
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.measure import label, regionprops
from scipy.ndimage import binary_erosion, binary_dilation, binary_closing
from skimage.morphology import ball
import json
from scipy.spatial.distance import cdist
import numpy as np

In [10]:
# def find_peak(semantic_mask):
#     distance_map = ndimage.distance_transform_edt(semantic_mask)
#     local_maxi = peak_local_max(distance_map, min_distance=4, threshold_abs=0.2, exclude_border=False)
#     return distance_map, local_maxi

def generate_distance_map(semantic_mask):
    distance_map = ndimage.distance_transform_edt(semantic_mask)
    return distance_map

def generate_local_maxi(semantic_mask):
    local_maxi = peak_local_max(semantic_mask, min_distance=4, threshold_abs=0.2, exclude_border=False)
    return local_maxi

In [11]:
semantic_path = f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT.mrc'
semantic_mask = get_tomo(semantic_path)
# distance_map, local_maxi = find_peak(semantic_mask)

In [12]:
distance_map = generate_distance_map(semantic_mask)

In [13]:
# save_tomo(distance_map, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/distance_map.mrc')
center_distance_map = np.zeros_like(semantic_mask, dtype=np.int8)
center_distance_map[distance_map > 4] = 1
instance_label = label(center_distance_map)
# save_tomo(instance_label, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/instance_label.mrc')
# instance_label = label(center_distance_map)

In [ ]:
flat = instance_label.ravel()

# 1) 统计每个标签的像素数（比 unique + counts 更快）
counts = np.bincount(flat)
# 2) 挑出“像素数 >= 300” 的大标签
large_labels = np.nonzero(counts >= 300)[0]

# 3) 一次性保留大标签，其它全部置 0
#    用映射表：下标是原标签，值是要映射到的新标签（不是大标签就映射到 0）
mapping = np.zeros_like(counts, dtype=instance_label.dtype)
mapping[large_labels] = large_labels
# 4) 重映射一把，底层 C 实现，速度极快
flat[:] = mapping[flat]

new_instance_label = flat.reshape(instance_label.shape)

# save_tomo(new_instance_label, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/new_instance_label.mrc')


In [16]:
# new_distance_map = generate_distance_map(new_instance_label)
# save_tomo(new_distance_map, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/new_distance_map.mrc')
local_maxi = generate_local_maxi(distance_map)
print(local_maxi)
# print(distance_map.shape)

[[ 47 860   0]
 [ 66 833  75]
 [ 56 713  16]
 [ 57 732  29]
 [ 58 834 106]
 [ 63 738   0]
 [ 65 784  34]
 [ 65 867 103]
 [ 65 888 119]
 [ 66 819  63]
 [ 66 823  66]
 [ 66 838  79]
 [ 66 877 111]
 [ 66 881 114]
 [ 67  42 722]
 [ 74 900   0]
 [ 85 165 278]
 [ 86 156 248]
 [ 86 158 254]
 [ 86 160 260]
 [ 86 164 274]
 [ 91 144 278]
 [ 94 158 203]
 [ 94 159 207]
 [105 160 290]
 [ 56 695   4]
 [ 57 788  72]
 [ 57 798  80]
 [ 58 839 109]
 [ 58 845 113]
 [ 58 850 116]
 [ 64 790  39]
 [ 65 807  53]
 [ 65 812  57]
 [ 65 857  95]
 [ 65 862  99]
 [ 66 845  85]
 [ 70  58 756]
 [ 71  60 760]
 [ 74  74 795]
 [ 85 167 285]
 [ 86 162 267]
 [ 89 151 305]
 [ 90 148 291]
 [ 90 149 295]
 [ 90 181 216]
 [ 90 182 220]
 [ 90 183 225]
 [ 56 703   9]
 [ 56 707  12]
 [ 56 718  19]
 [ 56 722  22]
 [ 56 762  51]
 [ 57 744  38]
 [ 57 748  41]
 [ 57 754  45]
 [ 57 768  56]
 [ 57 773  60]
 [ 57 778  64]
 [ 57 782  67]
 [ 57 793  76]
 [ 57 803  84]
 [ 57 807  87]
 [ 57 811  90]
 [ 60 878 135]
 [ 60 884 139]
 [ 61 888 

In [17]:
markers = np.zeros_like(semantic_mask, dtype=int)
for i, peak in enumerate(local_maxi):
    markers[tuple(peak)] = i + 1  # 标记每个局部最大值为唯一的实例ID
save_tomo(markers, 
          '/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/keypoints.mrc')

In [18]:
def euclidean_distance_matrix(points):
    """
    计算三维空间中所有点之间的欧几里得距离，并返回一个距离矩阵
    """
    diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
    dist_matrix = np.linalg.norm(diff, axis=2)
    return dist_matrix

def tsp_3d(points):
    """
    基于贪心算法计算在三维空间中按最短距离连接点的顺序，返回排序后的points列表
    尝试每个点作为起点，找到最短的路径。
    """
    n = len(points)
    dist_matrix = euclidean_distance_matrix(points)
    
    # 初始化最小路径和排序
    min_total_length = float('inf')
    best_order = []

    # 尝试每个点作为起点
    for start_idx in range(n):
        visited = [False] * n
        order = []
        current_idx = start_idx
        visited[current_idx] = True
        order.append(points[current_idx])
        total_length = 0

        for _ in range(n - 1):
            # 当前点到其他未访问点的距离
            dist_to_current = dist_matrix[current_idx]
            
            # 选择最近的未访问点
            nearest_idx = np.argmin([dist_to_current[i] if not visited[i] else np.inf for i in range(n)])
            
            # 更新路径
            current_idx = nearest_idx
            visited[current_idx] = True
            order.append(points[current_idx])
            total_length += dist_to_current[current_idx]

        # 比较得到最小路径
        if total_length < min_total_length:
            min_total_length = total_length
            best_order = order

    return best_order

In [23]:
def find_instance_for_keypoints(instance_mask, keypoints_list):
    """
    根据instance_mask为每个关键点分配对应的实例ID
    """
    instance_ids = []
    for kp in keypoints_list:
        z, y, x = kp
        instance_id = instance_mask[int(z), int(y), int(x)]
        instance_ids.append(instance_id)
    return instance_ids

def sort_keypoints_by_distance(keypoints):
    """
    根据最短距离排序关键点，使点从头到尾连成的距离最短
    使用贪心算法，选择最近的点连接。
    """
    # 初始化第一个点
    sorted_keypoints = [keypoints[0]]
    keypoints_remaining = set(map(tuple, keypoints[1:]))  # 剩余的点
    current_point = keypoints[0]

    while keypoints_remaining:
        # 计算当前点到剩余点的距离
        distances = cdist([current_point], list(keypoints_remaining))
        # 找到最近的点
        next_point = list(keypoints_remaining)[np.argmin(distances)]
        sorted_keypoints.append(next_point)
        keypoints_remaining.remove(tuple(next_point))
        current_point = next_point
    
    return np.array(sorted_keypoints)

def calculate_total_length(sorted_keypoints):
    """
    计算按最短路径排序后的关键点总长度
    """
    length = 0
    for i in range(1, len(sorted_keypoints)):
        length += np.linalg.norm(sorted_keypoints[i] - sorted_keypoints[i - 1])
    return length

def save_results_to_json(instances_info, output_json_path):
    """
    将实例信息保存到JSON文件
    """
    with open(output_json_path, 'w') as f:
        json.dump(instances_info, f, indent=4)

def process_instance_masks(instance_mask, keypoints_list, output_json_path):
    """
    处理实例mask和关键点，生成最终的JSON结果
    """
    # 1. 为每个关键点分配实例ID
    instance_ids = find_instance_for_keypoints(instance_mask, keypoints_list)

    # 2. 按实例ID分组关键点
    instance_keypoints = {}
    for idx, instance_id in enumerate(instance_ids):
        instance_keypoints.setdefault(instance_id, []).append(keypoints_list[idx])

    # 2.1 删除 ID 为 0 的组（通常代表背景）
    instance_keypoints.pop(0, None)

    # 2.2 丢弃点数少于 3 的组
    instance_keypoints = {
        inst_id: pts
        for inst_id, pts in instance_keypoints.items()
        if len(pts) >= 3
    }

    # 3. 对每个实例的关键点按最短路径排序，并收集信息
    instances_info = []
    # 按原始 ID 排序（可选），然后重新编号
    for new_id, (orig_id, keypoints) in enumerate(sorted(instance_keypoints.items()), start=1):
        # 假设 tsp_3d 就是 sort_keypoints_by_distance
        sorted_kps = tsp_3d(np.array(keypoints))  
        total_length = calculate_total_length(sorted_kps)

        instances_info.append({
            "id": new_id,  # 重新从 1 开始编号
            # "orig_id": int(orig_id),  # 如果需要保留原始 ID，可以加上这行
            "points": [[float(z), float(y), float(x)] for z, y, x in sorted_kps],
            "length": float(total_length)
        })

    # 4. 保存结果到 JSON 文件
    save_results_to_json(instances_info, output_json_path)

In [24]:
output_json_path = '/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/mt_point.json'
process_instance_masks(new_instance_label, local_maxi, output_json_path)
